## This notebook compares CORA water levels with observations at selected NOAA CO-OPS NWLON stations. CORA data is stored in an S3 bucket and made freely available via NOAA's Open Data Dissemination (NODD) service.

Access all the data in the S3 bucket here: https://noaa-nos-cora-pds.s3.amazonaws.com/index.html

*Parts of the code in this notebook were guided by **Signell, R. (2024). Reproducible Notebooks/hurricane-ike-water-levels. https://github.com/reproducible-notebooks/hurricane-ike-water-levels/tree/master***

_ _

### Input the IDs of the NWLON stations at which to compare CORA with observations
#### We will use Charleston, SC (8665530) and Pilots Station East, S.W. Pass, LA (8760922) in this example. <br> Note that observations from Charleston are assimilated into the model, but observations from Pilots Station East, S.W. Pass are not.

In [ ]:
NWLON_IDs = [8665530, 8760922]  # Charleston SC and Pilots Station East, S.W. Pass LA

### Import packages ###

In [ ]:
import intake
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import zarr

### Initialize and list the available CORA datasets ###
Description of datasets: <br>
**CORA-{domain}-fort.63-{chunking}**: Hourly water levels <br>
**CORA-{domain}-swan_TPS.63-{chunking}**: Hourly spectral peak wave periods <br>
**CORA-{domain}-swan_DIR.63-{chunking}**: Hourly mean wave directions <br>
**CORA-{domain}-swan_HS.63-{chunking}**: Hourly significant wave heights <br>
**CORA-{domain}-Grid-{chunking}**: Hourly water levels interpolated from model nodes to a uniform 500 meter resolution grid <br>

Where: <br>
**{domain}** represents the domain. Options are **GEC** or **PAC**. <br>
**{chunking}** represents the chunking strategy, which affects data loading speed. Use **timeseries** if you are interested in long timeseries at a few points, use **map** if you are interested in many points for a few time periods (up to a few days).

In [ ]:
if int(zarr.__version__[0]) > 2:  ## There are some issues with zarr 2 vs. zarr 3, so two catalogs have been created depdning on the zarr version you are using.
    catalog_file = 'CORA_intake_zarr3.yml'
else:
    catalog_file = 'CORA_intake.yml'

catalog = intake.open_catalog("s3://noaa-nos-cora-pds/"+catalog_file,storage_options={'anon':True})
list(catalog)

### Initialize CORA-GEC water levels (see full dataset [here](https://noaa-nos-cora-pds.s3.amazonaws.com/index.html#cora_gec/native_grid/water_levels/)). ###
#### This dataset contains hourly water levels from 1979-2022 in the GEC domain. The dataset is multiple TB in size and cannot be loaded into memory on a standard computer. We use the xarray package to "lazily" load the data - initializing it and only loading the pieces that we need later. Note that we use the **timeseries** chunking here, since we are interested in a long timeseries at a few points.

In [ ]:
ds = catalog["CORA-GEC-fort.63-timeseries"].to_dask()
ds

### For each NWLON station, grab observations and CORA water levels at the nearest node 
#### We will limit the comparison to January 2018 in this example.

In [ ]:
# Define a haversine function so we can compute distance to the gauge.
def haversine(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    R = 6371 
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    km = R * c
    return km 

comp_list = []
for ID in NWLON_IDs:
    # Grab the observations from the CO-OPS API #
    url = ('https://api.tidesandcurrents.noaa.gov/api/prod/datagetter?'+
            'begin_date=20180101&end_date=20180131&station='+str(ID)+'&product=hourly_height'+
            '&datum=MSL&time_zone=gmt&units=metric&format=json')
    r_obs = requests.get(url)
    lat_station = float(r_obs.json()['metadata']['lat'])
    lon_station = float(r_obs.json()['metadata']['lon'])
    obs1 = pd.DataFrame(r_obs.json()['data'])
    obs = pd.DataFrame({'time':pd.to_datetime(obs1['t']),'val':obs1['v'].astype(float)})
    
    # Grab CORA data from the nearest node to the station #
    d = haversine(lat_station,lon_station,ds.y,ds.x)
    closest_node = int(np.argmin(d.data))
    cora = ds['zeta'].sel(node=closest_node,time=slice("2018-1-1","2018-1-31"))
    
    # Create a comparison DataFrame #
    comp_station = pd.DataFrame({'time':obs['time'],'obs':obs['val'],'cora':cora.values})
    comp_list.append(comp_station)

### Plot observations vs. CORA at each of the NWLON stations
#### Note that CORA water levels are in units of mm, while NWLON observations are in units of m. We will convert CORA to m for the plot.

In [ ]:
fig,ax = plt.subplots(len(NWLON_IDs),1,figsize=(10,len(NWLON_IDs)*2),sharex=True)
for n in range(len(NWLON_IDs)):
    o = ax[n].plot(comp_list[n]['time'],comp_list[n]['obs'],'k',zorder=2)
    c = ax[n].plot(comp_list[n]['time'],comp_list[n]['cora']/1000,'r',zorder=2)
    ax[n].set_ylabel('Water level (m, MSL)',fontsize=8)
    ax[n].grid('on',linestyle='--')
    ax[n].text(0.01,0.98,NWLON_IDs[n],ha='left',va='top',transform=ax[n].transAxes)
fig.legend([o[0],c[0]],['Observations','CORA'],fontsize=10)
fig.autofmt_xdate()